# Aprendizado de Máquina — Lista prática E3

## NLP + Classificação (Aplicação)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Última lista do curso, e a única sobre dados que não chegam em forma de tabela.
Vai aparecer quase tudo: representação, esparsidade, três classificadores, e a
escolha da métrica num problema desbalanceado.

E um resultado que contraria o conselho mais repetido em processamento de texto:

> **trocar contagens por TF-IDF *piora* dois dos três classificadores aqui. A
> transformação certa depende do modelo que vem depois dela.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import os

import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — do texto à matriz esparsa

O `spam.csv` é uma coleção de mensagens de SMS rotuladas. Ele foi exportado com
três colunas vazias sobrando e num encoding antigo — nada disso é acidente do
nosso arquivo, é como dados de texto costumam chegar.

In [ ]:
_nome = "spam.csv"
_local = os.path.join("..", "..", "recursos", "dados", _nome)   # repositorio clonado
_url = ("https://raw.githubusercontent.com/HugoCarvalhoUFRJ/ap-maq/"
        "refs/heads/refactoring-baby/recursos/dados/") + _nome  # fallback (ex.: Colab)
_fonte = _local if os.path.exists(_local) else _url

mensagens = pd.read_csv(_fonte, encoding="latin-1")           # (a) o arquivo NAO e utf-8
print("colunas do arquivo:", list(mensagens.columns))

mensagens = mensagens.iloc[:, :2]
mensagens.columns = ["rotulo", "texto"]
y = (mensagens["rotulo"] == "spam").astype(int).values        # (b)

print(f"n = {len(mensagens)}, spam = {y.mean():.4f} ({int(y.sum())} mensagens)")

In [ ]:
X_tr, X_te, y_tr, y_te = skm.train_test_split(
    mensagens["texto"].values, y, test_size=0.3,
    random_state=2026, stratify=y)                            # (a)

vetorizador = CountVectorizer()
Xc = vetorizador.fit_transform(X_tr)                          # (b) ajuste SO no treino

n_linhas, n_termos = Xc.shape
densidade = 100 * Xc.nnz / (n_linhas * n_termos)              # (c)

print(f"matriz: {Xc.shape}   nao-zeros: {Xc.nnz}   densidade: {densidade:.4f}%")
print(f"memoria esparsa: {(Xc.data.nbytes + Xc.indices.nbytes + Xc.indptr.nbytes) / 1e6:.2f} MB")
print(f"memoria densa:   {n_linhas * n_termos * 8 / 1e6:.1f} MB")

Deve imprimir as cinco colunas do arquivo (`v1`, `v2` e três `Unnamed`),
`n = 5572, spam = 0.1341 (747 mensagens)`, e:

```
matriz: (3900, 7168)   nao-zeros: 51409   densidade: 0.1839%
memoria esparsa: 0.63 MB
memoria densa:   223.6 MB
```

**Menos de duas entradas em mil são não nulas**, e o formato esparso ocupa 355
vezes menos memória. Não é uma otimização: guardar a matriz densa de um corpus
dez vezes maior já não caberia na memória de um computador comum.

Note também `encoding="latin-1"`: o arquivo tem bytes que não formam UTF-8
válido, e ler sem esse argumento levanta `UnicodeDecodeError`. E o
`stratify=y` importa aqui mais do que de costume — com 13,4% de positivos, uma
divisão sem estratificação pode dar dobras com prevalências bem diferentes.

Veja quais termos dominam a matriz, e quantos aparecem uma única vez.

In [ ]:
contagens = np.asarray(Xc.sum(axis=0)).ravel()
vocabulario = vetorizador.get_feature_names_out()
ordem = np.argsort(-contagens)                                # (a) do mais frequente ao menos

print("10 termos mais frequentes:")
print([(vocabulario[i], int(contagens[i])) for i in ordem[:10]])

# em quantos DOCUMENTOS cada termo aparece (nao quantas vezes ao todo)
n_docs = np.asarray((Xc > 0).sum(axis=0)).ravel()
raros = int((n_docs == 1).sum())                              # (b) termos em um unico documento
print(f"\ntermos que aparecem em 1 documento so: {raros} de {n_termos} "
      f"({100 * raros / n_termos:.1f}%)")

Deve imprimir

```
10 termos mais frequentes:
[('to', 1556), ('you', 1545), ('the', 888), ('and', 678), ('in', 635),
 ('is', 607), ('me', 570), ('it', 521), ('my', 519), ('for', 502)]

termos que aparecem em 1 documento so: 3867 de 7168 (53.9%)
```

As duas pontas da distribuição de frequências, e as duas são inúteis por motivos
opostos.

No topo estão **palavras funcionais** — *to, you, the, and* — que aparecem em
quase toda mensagem e por isso não distinguem spam de não-spam. É contra elas que
o idf do Exercício 1 da Lista Teórica E3 age, e é o que uma lista de *stop words*
removeria à mão.

Na cauda, **mais da metade do vocabulário aparece em um único documento**. Cada
um desses 3 867 termos é um preditor perfeito no treino (se está numa mensagem de
spam, prevê spam com certeza) e ruído puro fora dele. É contra eles que o
`min_df` age.

Essa forma de distribuição — pouquíssimos termos muito frequentes, uma cauda
enorme de termos raríssimos — é a **lei de Zipf**, e é universal em linguagem.

---
## Exercício 2 — `min_df` como regularização

Se metade do vocabulário aparece uma vez só, cortar essa cauda deveria ajudar.
Meça quanto ela pesa.

In [ ]:
for m in (1, 2, 5, 10):
    v = CountVectorizer(min_df=m).fit(X_tr)                   # (a)
    print(f"min_df={m:2d}: {len(v.get_feature_names_out()):5d} termos")

Deve imprimir:

```
min_df= 1:  7168 termos
min_df= 2:  3301 termos
min_df= 5:  1418 termos
min_df=10:   770 termos
```

Exigir que o termo apareça em **dois** documentos em vez de um já corta o
vocabulário pela metade (7 168 → 3 301) — coerente com os 53,9% do Exercício 1.
Com `min_df=5` sobra um quinto, e com 10, um décimo.

Vale insistir no ponto do Exercício 3(c) da Lista Teórica E3: `min_df` **é um
hiperparâmetro de regularização**, ainda que a API o apresente como opção do
vetorizador. Ele controla a complexidade do modelo exatamente como o `alpha` da
Ridge, e por isso pertence ao `Pipeline` e à grade de busca — não a uma decisão
tomada uma vez, no começo, olhando os dados todos.

---
## Exercício 3 — seis combinações

Duas representações (contagens e TF-IDF) e três classificadores lineares. Meça
acurácia, $F_1$ e — o que mais importa num filtro de spam — os falsos positivos e
falsos negativos separadamente.

In [ ]:
representacoes = [("contagens", CountVectorizer()),           # (a)
                  ("TF-IDF",    TfidfVectorizer())]           # (b)

classificadores = [("MultinomialNB", MultinomialNB()),
                   ("logistica",     LogisticRegression(max_iter=2000)),
                   ("LinearSVC",     LinearSVC(dual=True, max_iter=5000))]

for nome_rep, vetor in representacoes:
    for nome_clf, modelo in classificadores:
        tubo = Pipeline([("vetor", vetor), ("clf", modelo)]).fit(X_tr, y_tr)
        pred = tubo.predict(X_te)
        vn, fp, fn, vp = confusion_matrix(y_te, pred).ravel()  # (c)
        print(f"{nome_rep:10s} {nome_clf:14s} acuracia {tubo.score(X_te, y_te):.4f}"
              f"  F1 {f1_score(y_te, pred):.4f}   FP {fp:3d}  FN {fn:3d}")

Deve imprimir:

```
contagens  MultinomialNB  acuracia 0.9856  F1 0.9450   FP   6  FN  18
contagens  logistica      acuracia 0.9815  F1 0.9264   FP   2  FN  29
contagens  LinearSVC      acuracia 0.9844  F1 0.9387   FP   1  FN  25
TF-IDF     MultinomialNB  acuracia 0.9605  F1 0.8272   FP   0  FN  66
TF-IDF     logistica      acuracia 0.9653  F1 0.8520   FP   1  FN  57
TF-IDF     LinearSVC      acuracia 0.9856  F1 0.9442   FP   3  FN  21
```

**O TF-IDF piora dois dos três classificadores**, e não por pouco: o
`MultinomialNB` cai de $0{,}9856$ para $0{,}9605$ e a logística de $0{,}9815$
para $0{,}9653$. Só o `LinearSVC` melhora.

A razão é diferente em cada caso, e vale entender as duas.

O **naive Bayes multinomial** modela contagens: a verossimilhança dele é a de um
sorteio multinomial de palavras, e $N_{wc}$ na fórmula do Exercício 2 da Lista
Teórica E3 é *uma contagem*. Alimentá-lo com pesos TF-IDF fracionários e
normalizados quebra a suposição do modelo — ele continua funcionando, porque a
ordenação dos escores se preserva em boa medida, mas perde.

A **logística e a SVM** não se importam com a interpretação das colunas, só com
a escala. Aí a normalização $\ell_2$ do TF-IDF ajuda ou atrapalha conforme a
penalização: a SVM, que é sensível à norma dos vetores (a margem é medida em
distância), ganha; a logística com $\ell_2$ padrão, aqui, perde.

**E olhe a coluna FP.** Num filtro de spam, um falso positivo é uma mensagem
legítima jogada fora — muito pior que um spam que passou. Por esse critério o
vencedor é `contagens + LinearSVC`, com **1 falso positivo** contra 6 do
`MultinomialNB`, apesar de ter acurácia menor. É o Exercício 2 da Lista Teórica
09 outra vez: quando os custos são assimétricos, a métrica simétrica escolhe
errado.

> **Sua vez.** O `TF-IDF + MultinomialNB` tem **zero** falsos positivos. Isso o
> torna a melhor escolha para um filtro de spam? Olhe a coluna FN antes de
> responder, e pense em qual seria o desempenho de um classificador que responde
> sempre "não é spam".

---
## Exercício 4 — o que o modelo aprendeu

Um modelo linear sobre texto é diretamente legível: cada coluna é uma palavra, e
o coeficiente dela diz o quanto ela empurra para spam ou para não-spam.

In [ ]:
tubo = Pipeline([("vetor", TfidfVectorizer()),
                 ("clf", LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)

palavras = tubo.named_steps["vetor"].get_feature_names_out()
coef = tubo.named_steps["clf"].coef_[0]                       # (a) uma linha so: problema binario

ordem = np.argsort(coef)                                      # (b) do mais negativo ao mais positivo

print("mais indicativas de SPAM:")
print([(palavras[i], round(coef[i], 3)) for i in ordem[::-1][:10]])    # (c)
print("\nmais indicativas de NAO-SPAM:")
print([(palavras[i], round(coef[i], 3)) for i in ordem[:10]])

Deve imprimir:

```
mais indicativas de SPAM:
[('call', 4.087), ('txt', 3.857), ('free', 3.533), ('stop', 3.291),
 ('text', 2.933), ('mobile', 2.796), ('uk', 2.693), ('claim', 2.682),
 ('reply', 2.623), ('chat', 2.563)]

mais indicativas de NAO-SPAM:
[('me', -2.404), ('my', -2.145), ('ok', -1.567), ('gt', -1.543),
 ('lt', -1.534), ('it', -1.503), ('that', -1.478), ('ll', -1.395),
 ('how', -1.383), ('but', -1.371)]
```

O modelo aprendeu exatamente o que um humano diria: spam de SMS pede que você
**ligue** (*call*), mande **mensagem** (*txt*, *text*, *reply*), oferece coisas
**grátis** (*free*) e manda **reivindicar** um prêmio (*claim*).

Do outro lado estão marcas de conversa pessoal: pronomes de primeira pessoa
(*me*, *my*), interjeições (*ok*) e conjunções (*that*, *but*). Ninguém escreve
"*my*" numa propaganda em massa.

Repare no `gt` e no `lt`: são os restos de `&gt;` e `&lt;` — entidades HTML que
sobreviveram à exportação e viraram tokens. Eles aparecem em mensagens pessoais
(emoticons como `<3`) e o modelo os usa. **Não é sinal, é artefato de
codificação do arquivo**, e num sistema em produção seria um problema: se o
pipeline de coleta mudar e passar a decodificar o HTML corretamente, essas duas
colunas somem e o desempenho cai sem aviso.

É a última lição do curso, e ela não é sobre nenhum algoritmo: **olhar os
coeficientes é barato e revela coisas que nenhuma métrica revela.** A acurácia de
$0{,}9653$ não diria que dois dos preditores mais úteis do modelo são um erro de
*encoding*.

> **Sua vez.** Monte a busca completa — `ngram_range` em `[(1,1), (1,2)]`,
> `min_df` em `[1, 2, 5]` e o `C` da logística — num único `GridSearchCV` sobre o
> `Pipeline`, com `scoring="f1"`. Compare o resultado com as seis linhas do
> Exercício 3.

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | densidade de 0,18%: o formato esparso ocupa **355×** menos memória |
| 1 | 53,9% dos termos aparecem em **um único** documento |
| 2 | exigir 2 documentos em vez de 1 corta o vocabulário pela metade |
| 3 | o TF-IDF **piora** o naive Bayes (0,9856 → 0,9605) e a logística (0,9815 → 0,9653) |
| 3 | o melhor por falsos positivos é `contagens + LinearSVC` (FP = 1), que não é o melhor em acurácia |
| 4 | dois dos preditores mais fortes de não-spam, `gt` e `lt`, são artefato de codificação |

---

## Encerramento

Esta lista fecha o curso, e vale olhar para trás pelo que as catorze listas
mediram, não pelo que os métodos prometem.

O erro de treino é otimista, e o otimismo tem fórmula. A validação cruzada
estima risco, e quem escolheu não pode reportar. Cotas superiores não atingidas
não são cotas erradas. O critério que separa vazamento grave de leve é olhar o
$Y$, não aprender dos dados. Ordenar bem não é estimar bem. Um aviso verdadeiro
("o naive Bayes é mal calibrado") pode ser condicional a uma hipótese que os seus
dados não satisfazem — e conferir custou uma linha.

Em todas essas vezes, o que resolveu foi a mesma coisa: **medir, em vez de
repetir.**